# FLUX Model Download & Compression (Disk-Optimized for Kaggle)

**Objective:** Download Flux-Apex-V1.flx from HuggingFace and compress it for easier distribution.

**What this notebook does:**
1. Downloads Flux-Apex-V1.flx (~17.4GB) from HuggingFace Hub
2. **Streams compression while freeing disk space** (Kaggle-optimized)
3. Optionally splits into smaller chunks for easier download
4. Provides download links for Kaggle/Colab output

**Kaggle Optimization:** This notebook handles the 20GB disk limit by:
- Loading the model into memory
- Deleting the original file to free ~17GB
- Writing the compressed version (~10GB)

**Expected compression:** ~40-50% reduction with gzip

In [ ]:
"""Cell 1: Environment Setup"""

import os
import sys
import gc
import gzip
import lzma
import shutil
from pathlib import Path

# Environment detection
if Path('/kaggle').exists():
    ROOT = Path('/kaggle/working/FLUX')
    OUTPUT_DIR = Path('/kaggle/working')
    ENVIRONMENT = 'kaggle'
    # Clone repo for utilities
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /kaggle/working/FLUX')
elif Path('/content').exists():
    ROOT = Path('/content/FLUX')
    OUTPUT_DIR = Path('/content')
    ENVIRONMENT = 'colab'
    if not ROOT.exists():
        os.system('git clone https://github.com/Unseengap/FLUX.git /content/FLUX')
else:
    ROOT = Path('/Users/admin/Desktop/flux')
    OUTPUT_DIR = ROOT / 'checkpoints'
    ENVIRONMENT = 'local'

sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print(f"✓ Environment: {ENVIRONMENT}")
print(f"✓ Root: {ROOT}")
print(f"✓ Output directory: {OUTPUT_DIR}")

In [ ]:
"""Cell 2: Setup Utilities & HuggingFace Token"""

from flux_utils import PhaseLogger, _resolve_hf_token

log = PhaseLogger(phase=300)  # Phase 300 = Download/Compress utility
log.separator("FLUX Download & Compression")

# Resolve HuggingFace token
try:
    hf_token = _resolve_hf_token()
    log.success("HuggingFace token resolved")
except:
    hf_token = None
    log.warning("No HF token - may have limited download speed")

# Check disk space
if ENVIRONMENT in ['kaggle', 'colab']:
    import subprocess
    result = subprocess.run(['df', '-h', str(OUTPUT_DIR)], capture_output=True, text=True)
    print("\nDisk space available:")
    print(result.stdout)

In [ ]:
"""Cell 3: Download Flux-Apex-V1.flx from HuggingFace"""

log.cell_start("Cell 3 — Download Model")

from huggingface_hub import hf_hub_download

CHECKPOINT_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'

if CHECKPOINT_PATH.exists():
    log.info(f"Model already exists: {CHECKPOINT_PATH}")
    log.info(f"Size: {CHECKPOINT_PATH.stat().st_size / 1e9:.2f} GB")
else:
    log.info("Downloading Flux-Apex-V1.flx from HuggingFace...")
    log.info("This may take 10-30 minutes depending on connection speed.")
    
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    downloaded = hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=hf_token,
    )
    
    log.success(f"Downloaded: {downloaded}")
    log.info(f"Size: {Path(downloaded).stat().st_size / 1e9:.2f} GB")

original_size = CHECKPOINT_PATH.stat().st_size
log.cell_end("Cell 3 — Download Model", "PASS")

In [ ]:
"""Cell 4: Compression Options & Disk Space Check"""

import subprocess

# Choose compression method
# For Kaggle (20GB disk), use 'gzip' - fastest and good enough compression
# 'lzma' gives better compression but is slower
COMPRESSION_METHOD = 'gzip'  # Best for Kaggle due to speed
CHUNK_SIZE_GB = 0  # Disable chunking to save disk space on Kaggle

# Check available disk space
def get_free_space_gb(path: Path) -> float:
    """Get free disk space in GB."""
    result = subprocess.run(['df', '-B1', str(path)], capture_output=True, text=True)
    lines = result.stdout.strip().split('\n')
    if len(lines) >= 2:
        parts = lines[1].split()
        return int(parts[3]) / 1e9  # Available space
    return 0

free_space = get_free_space_gb(OUTPUT_DIR)
model_size_gb = original_size / 1e9

print(f"\n📊 Disk Space Analysis:")
print(f"  Free space: {free_space:.1f} GB")
print(f"  Model size: {model_size_gb:.2f} GB")
print(f"  Est. compressed: ~{model_size_gb * 0.55:.1f} GB (gzip)")

# Determine if we need disk-constrained mode
DISK_CONSTRAINED = free_space < (model_size_gb * 0.6)  # Need ~60% for compressed output

if DISK_CONSTRAINED:
    print(f"\n⚠️  DISK-CONSTRAINED MODE ENABLED")
    print(f"  Will delete original file during compression to free space")
else:
    print(f"\n✓ Sufficient disk space for standard compression")

print(f"\nCompression settings:")
print(f"  Method: {COMPRESSION_METHOD}")
print(f"  Mode: {'Disk-constrained (delete-during-compress)' if DISK_CONSTRAINED else 'Standard'}")

In [ ]:
"""Cell 5: Compress the Model File (Disk-Optimized)"""

log.cell_start("Cell 5 — Compression")

import time

def compress_disk_constrained(input_path: Path, output_path: Path, method: str = 'gzip') -> Path:
    """
    Compress large file on disk-constrained systems (Kaggle/Colab).
    
    Strategy:
    1. Read entire file into RAM (Kaggle has 16GB+)
    2. Delete original file to free disk space  
    3. Write compressed output
    
    This allows compression when free_space < file_size + compressed_size
    """
    out = output_path.with_suffix('.flx.gz' if method == 'gzip' else '.flx.xz')
    file_size = input_path.stat().st_size
    
    print(f"  📖 Reading {file_size/1e9:.2f} GB into memory...")
    
    # Read entire file into memory
    with open(input_path, 'rb') as f:
        file_data = f.read()
    
    print(f"  ✓ File loaded into memory ({len(file_data)/1e9:.2f} GB)")
    
    # Delete original file to free disk space
    print(f"  🗑️ Deleting original to free {file_size/1e9:.2f} GB disk space...")
    os.remove(input_path)
    gc.collect()
    
    # Verify space freed
    free_after = get_free_space_gb(output_path.parent)
    print(f"  ✓ Freed disk space. Now available: {free_after:.1f} GB")
    
    # Compress and write
    print(f"  🗜️ Compressing with {method}...")
    
    if method == 'gzip':
        with gzip.open(out, 'wb', compresslevel=6) as f_out:  # Level 6 = good balance
            f_out.write(file_data)
    else:  # lzma
        with lzma.open(out, 'wb', preset=4) as f_out:  # Lower preset for speed
            f_out.write(file_data)
    
    # Free memory
    del file_data
    gc.collect()
    
    return out


def compress_standard(input_path: Path, output_path: Path, method: str = 'gzip') -> Path:
    """Standard streaming compression when disk space is sufficient."""
    if method == 'gzip':
        out = output_path.with_suffix('.flx.gz')
        with open(input_path, 'rb') as f_in:
            with gzip.open(out, 'wb', compresslevel=6) as f_out:
                shutil.copyfileobj(f_in, f_out, length=1024*1024*64)
    elif method == 'lzma':
        out = output_path.with_suffix('.flx.xz')
        with open(input_path, 'rb') as f_in:
            with lzma.open(out, 'wb', preset=4) as f_out:
                shutil.copyfileobj(f_in, f_out, length=1024*1024*64)
    return out


output_base = OUTPUT_DIR / 'Flux-Apex-V1'

log.info(f"Compressing with {COMPRESSION_METHOD}...")
if DISK_CONSTRAINED:
    log.warning("Using disk-constrained mode (will delete original during compression)")

start_time = time.time()

# Choose compression strategy based on disk space
if DISK_CONSTRAINED:
    compressed_path = compress_disk_constrained(CHECKPOINT_PATH, output_base, COMPRESSION_METHOD)
else:
    compressed_path = compress_standard(CHECKPOINT_PATH, output_base, COMPRESSION_METHOD)

elapsed = time.time() - start_time

compressed_size = compressed_path.stat().st_size
ratio = (1 - compressed_size / original_size) * 100

log.success(f"Compression complete!")
log.info(f"Output: {compressed_path}")
log.info(f"Original: {original_size / 1e9:.2f} GB")
log.info(f"Compressed: {compressed_size / 1e9:.2f} GB")
log.info(f"Reduction: {ratio:.1f}%")
log.info(f"Time: {elapsed/60:.1f} minutes")

# Show final disk space
final_free = get_free_space_gb(OUTPUT_DIR)
log.info(f"Disk space remaining: {final_free:.1f} GB")

log.cell_end("Cell 5 — Compression", "PASS")

In [ ]:
"""Cell 6: Verify & Report (Chunking Disabled for Kaggle)"""

log.cell_start("Cell 6 — Verification")

# Verify the compressed file is valid
print(f"\n🔍 Verifying compressed file...")

try:
    # Quick validation - read first and last few bytes
    if COMPRESSION_METHOD == 'gzip':
        with gzip.open(compressed_path, 'rb') as f:
            header = f.read(1024)  # Read first 1KB
            f.seek(0, 2)  # Seek to end to verify file integrity
        print(f"  ✓ Gzip file is valid")
    else:
        with lzma.open(compressed_path, 'rb') as f:
            header = f.read(1024)
            f.seek(0, 2)
        print(f"  ✓ LZMA file is valid")
except Exception as e:
    log.error(f"Compressed file validation failed: {e}")
    raise

# Report chunk info (even though we're not splitting)
chunk_paths = [compressed_path]  # Single file, no chunks

print(f"\n📦 Output Files:")
for cp in chunk_paths:
    print(f"  • {cp.name}: {cp.stat().st_size / 1e9:.2f} GB")

log.cell_end("Cell 6 — Verification", "PASS")

In [ ]:
"""Cell 7: Generate Download Summary"""

log.cell_start("Cell 7 — Summary")

print("\n" + "="*60)
print("  FLUX DOWNLOAD & COMPRESSION COMPLETE")
print("="*60)

print(f"\n📦 Original Model:")
print(f"   • Source: HuggingFace UnseenGAP/FLUX")
print(f"   • File: Flux-Apex-V1.flx")
print(f"   • Size: {original_size / 1e9:.2f} GB")
print(f"   • Status: {'DELETED (disk-constrained mode)' if DISK_CONSTRAINED else 'Available'}")

print(f"\n🗜️ Compressed Output:")
print(f"   • File: {compressed_path.name}")
print(f"   • Size: {compressed_size / 1e9:.2f} GB")
print(f"   • Reduction: {ratio:.1f}%")
print(f"   • Method: {COMPRESSION_METHOD}")

print(f"\n📂 Output Location: {OUTPUT_DIR}")

# Show disk space
free_space_now = get_free_space_gb(OUTPUT_DIR)
print(f"\n💾 Disk Space Remaining: {free_space_now:.1f} GB")

# Environment-specific instructions
if ENVIRONMENT == 'kaggle':
    print("\n📥 Kaggle Download:")
    print("   1. Click 'Save Version' in top right")
    print("   2. Select 'Save & Run All (Commit)'")
    print("   3. Go to Output tab after completion")
    print("   4. Download the .gz file")
    
elif ENVIRONMENT == 'colab':
    print("\n📥 Colab Download:")
    print("   Run the next cell to download directly")

log.cell_end("Cell 7 — Summary", "PASS")

In [ ]:
"""Cell 8: Download Files (Colab/Kaggle)"""

# For Colab - trigger browser download
if ENVIRONMENT == 'colab':
    from google.colab import files
    
    print("Initiating download...")
    files.download(str(compressed_path))
    print("\n✓ Download initiated!")

elif ENVIRONMENT == 'kaggle':
    print("📥 Kaggle Output Instructions:")
    print("="*50)
    print("1. Click 'Save Version' in the top right")
    print("2. Select 'Save & Run All (Commit)'")
    print("3. After completion, go to your notebook's Output tab")
    print("4. Download the compressed file")
    print("="*50)
    
    # Ensure file is in /kaggle/working for output
    kaggle_output = Path('/kaggle/working')
    final_output = kaggle_output / compressed_path.name
    
    if compressed_path != final_output and compressed_path.exists():
        # Move instead of copy to save disk space
        shutil.move(str(compressed_path), str(final_output))
        compressed_path = final_output
        print(f"\n✓ Moved to: {final_output}")
    else:
        print(f"\n✓ Ready at: {compressed_path}")
    
    print(f"\n📦 Output file: {compressed_path.name} ({compressed_size/1e9:.2f} GB)")

else:
    print(f"\n✓ Local file ready at:")
    print(f"  {compressed_path}")

In [ ]:
"""Cell 9: Decompression Instructions"""

print("\n" + "="*60)
print("  HOW TO DECOMPRESS")
print("="*60)

if COMPRESSION_METHOD == 'gzip':
    print(f"\n📋 Command Line:")
    print(f"   gunzip Flux-Apex-V1.flx.gz")
    print(f"\n   Or keep original:")
    print(f"   gunzip -k Flux-Apex-V1.flx.gz")
    
    print(f"\n🐍 Python:")
    print("""   import gzip, shutil
   with gzip.open('Flux-Apex-V1.flx.gz', 'rb') as f_in:
       with open('Flux-Apex-V1.flx', 'wb') as f_out:
           shutil.copyfileobj(f_in, f_out)""")

else:  # lzma
    print(f"\n📋 Command Line:")
    print(f"   xz -d Flux-Apex-V1.flx.xz")
    print(f"\n   Or keep original:")
    print(f"   xz -dk Flux-Apex-V1.flx.xz")
    
    print(f"\n🐍 Python:")
    print("""   import lzma, shutil
   with lzma.open('Flux-Apex-V1.flx.xz', 'rb') as f_in:
       with open('Flux-Apex-V1.flx', 'wb') as f_out:
           shutil.copyfileobj(f_in, f_out)""")

print("\n" + "="*60)
print("\n📎 After decompression, load the model with:")
print("""   from flux_model import FLUXModel
   model = FLUXModel.load('Flux-Apex-V1.flx')""")

In [ ]:
"""Cell 10: Final Status"""

print("\n📊 Final Disk Status:")
final_free = get_free_space_gb(OUTPUT_DIR)
print(f"  Free space: {final_free:.1f} GB")

print("\n📁 Files:")
if CHECKPOINT_PATH.exists():
    print(f"  • Original: {CHECKPOINT_PATH.name} ({CHECKPOINT_PATH.stat().st_size/1e9:.2f} GB)")
else:
    print(f"  • Original: DELETED (was {original_size/1e9:.2f} GB)")
    
print(f"  • Compressed: {compressed_path.name} ({compressed_size/1e9:.2f} GB)")

if DISK_CONSTRAINED:
    print("\n⚠️  Note: Original file was deleted during compression to save disk space.")
    print("   The compressed file contains all the data. Decompress to restore.")
else:
    print("\n💡 To free more space, delete the original file:")
    print(f"   os.remove('{CHECKPOINT_PATH}')")

---

## Usage Notes

### After Downloading:

1. **Decompress** using the commands shown in the previous cell
2. **Load the model**:
   ```python
   from flux_model import FLUXModel
   model = FLUXModel.load('Flux-Apex-V1.flx')
   ```

### Kaggle-Specific Notes:

This notebook is optimized for Kaggle's 20GB disk limit:
- **Disk-constrained mode**: When free space < 60% of model size, the original file is automatically deleted during compression
- **Memory usage**: The model is loaded into RAM (Kaggle has 16GB+) before deleting
- **Compression level**: Uses level 6 gzip (good balance of speed/ratio)
- **Expected output**: ~9-10GB compressed file

### Direct HuggingFace Alternative:

If you prefer not to compress, download directly:
```python
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id='UnseenGAP/FLUX',
    filename='checkpoints/Flux-Apex-V1.flx',
    local_dir='.',
)
```

### Model Info:
- **Size**: ~17.4 GB (uncompressed) / ~9-10 GB (gzip)
- **Parameters**: ~8.34B
- **Format**: PyTorch serialized (.flx)
- **Version**: 8.1-complete